# LeetCode 3463 — Check If Digits Are Equal in String After Operations II bằng Python

- Link: https://leetcode.com/problems/check-if-digits-are-equal-in-string-after-operations-ii/
- Độ khó: **Hard**
- Chủ đề: **Tam giác Pascal**, **hệ số nhị thức modulo 10**, **định lý Lucas** (mod 2, mod 5) + **định lý số dư Trung Hoa (CRT)**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def hasSameDigits(self, s: str) -> bool:
```

## 1. Mục tiêu học

1. Nhận ra phép "cộng hai số kề mod 10" lặp lại chính là **tam giác Pascal**: chữ số cuối = `Σ C(m, i) · s[i] (mod 10)` với `m = n − 2`.
2. Tính `C(m, i) mod 10` khi `m` tới `10^5` mà không dùng số nguyên khổng lồ.
3. Áp dụng **định lý Lucas** cho số nguyên tố 2 và 5, rồi ghép bằng **CRT** (`10 = 2 · 5`).
4. Cách thay thế: cập nhật `C(m, i)` dần dần, **tách riêng lũy thừa của 2 và 5** để chia được modulo 10.
5. So sánh với mô phỏng trực tiếp `O(n²)` (thuần Python và `numpy`).

## 2. Đề bài & ví dụ

Cho xâu chữ số `s`. Lặp: thay `s` bằng dãy `(s[i] + s[i+1]) mod 10` cho mọi cặp kề nhau, đến khi còn **2** chữ số. Trả về `True` nếu hai chữ số cuối bằng nhau.

**Ví dụ 1:** `"3902"` → `"292"` → `"11"` ⇒ `True`.

**Ví dụ 2:** `"34789"` → `"7157"` → `"862"` → `"48"` ⇒ `False`.

**Ràng buộc:** `3 ≤ |s| ≤ 10^5`.

In [1]:
examples = [("3902", True), ("34789", False)]

def simulate(s):
    a = list(map(int, s))
    while len(a) > 2:
        a = [(a[i] + a[i + 1]) % 10 for i in range(len(a) - 1)]
    return a

for s, exp in examples:
    print(s, '-> hai chữ số cuối', simulate(s), '| kỳ vọng', exp)

3902 -> hai chữ số cuối [1, 1] | kỳ vọng True
34789 -> hai chữ số cuối [4, 8] | kỳ vọng False


## 3. Kiến thức nền

### 3.1. Tam giác Pascal xuất hiện thế nào

Sau 1 lần: `x_i = s_i + s_{i+1}`. Sau 2 lần: `s_i + 2 s_{i+1} + s_{i+2}`. Sau `t` lần: `Σ_j C(t, j) s_{i+j}`. Còn 2 chữ số sau `t = n − 2` lần:

- chữ số đầu `= Σ_{i=0}^{m} C(m, i) · s[i] (mod 10)`
- chữ số sau `= Σ_{i=0}^{m} C(m, i) · s[i+1] (mod 10)`, với `m = n − 2`.

In [2]:
from math import comb

def by_binomial_bigint(s):
    d = list(map(int, s))
    m = len(d) - 2
    a = sum(comb(m, i) * d[i] for i in range(m + 1)) % 10
    b = sum(comb(m, i) * d[i + 1] for i in range(m + 1)) % 10
    return [a, b]

for s, _ in examples:
    print(s, simulate(s), by_binomial_bigint(s))
print('C(60, i) lớn cỡ', len(str(comb(60, 30))), 'chữ số; C(10^5, 5·10^4) có', comb(10**5, 5 * 10**4).bit_length(), 'bit')

3902 [1, 1] [1, 1]
34789 [4, 8] [4, 8]
C(60, i) lớn cỡ 18 chữ số; C(10^5, 5·10^4) có 99992 bit


### 3.2. Định lý Lucas

Với `p` nguyên tố, viết `n = (n_k … n_0)_p`, `k = (k_k … k_0)_p` thì `C(n, k) ≡ Π C(n_j, k_j) (mod p)` (quy ước `C(a, b) = 0` nếu `b > a`).

Đặc biệt `p = 2`: `C(n, k)` lẻ ⇔ `k & n == k` (mọi bit của `k` đều có trong `n`).

In [3]:
def comb_mod_p(n, k, p):
    res = 1
    while n or k:
        a, b = n % p, k % p
        if b > a:
            return 0
        res = res * comb(a, b) % p
        n //= p; k //= p
    return res

assert all(comb_mod_p(n, k, p) == comb(n, k) % p for p in (2, 3, 5, 7) for n in range(60) for k in range(n + 1))
assert all((comb(n, k) % 2 == 1) == ((k & n) == k) for n in range(200) for k in range(n + 1))
print('Lucas khớp math.comb với p = 2, 3, 5, 7 (n < 60); và C(n,k) lẻ <=> k & n == k.')

Lucas khớp math.comb với p = 2, 3, 5, 7 (n < 60); và C(n,k) lẻ <=> k & n == k.


### 3.3. Định lý số dư Trung Hoa cho `10 = 2 · 5`

Biết `x mod 2 = r2` và `x mod 5 = r5` thì `x mod 10` xác định duy nhất. Lập bảng 10 ô một lần.

In [4]:
CRT10 = [[None] * 5 for _ in range(2)]
for x in range(10):
    CRT10[x % 2][x % 5] = x
print(CRT10)

[[0, 6, 2, 8, 4], [5, 1, 7, 3, 9]]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay ví dụ 1 (`"3902"`, `m = 2`): hệ số `C(2, ·) = 1, 2, 1`.

- Chữ số đầu: `1·3 + 2·9 + 1·0 = 21 ≡ 1`.
- Chữ số sau: `1·9 + 2·0 + 1·2 = 11 ≡ 1` ⇒ bằng nhau ⇒ `True`.

Vấn đề còn lại: `C(m, i)` với `m ~ 10^5` là số có hàng chục nghìn chữ số — tính trực tiếp quá chậm. Ta chỉ cần nó **mod 10**.

In [5]:
print([comb(2, i) for i in range(3)], '->', (3 + 2 * 9 + 0) % 10, (9 + 0 + 2) % 10)

[1, 2, 1] -> 1 1


## 5. Cách 1 — Brute Force: mô phỏng đúng như đề

In [6]:
def same_simulate(s):
    a, b = simulate(s)
    return a == b

for s, exp in examples:
    print(same_simulate(s), '| kỳ vọng', exp)

True | kỳ vọng True
False | kỳ vọng False


**Độ phức tạp:** `O(n²)` ≈ `5·10^9` phép cộng với `n = 10^5` — quá chậm trong Python thuần.

## 6. Ý tưởng thuật toán / bước đột phá

1. Kết quả chỉ phụ thuộc `C(m, i) mod 10`, `i = 0..m`.
2. `C(m, i) mod 2` bằng bit (`i & m == i`); `C(m, i) mod 5` bằng Lucas trên các chữ số cơ số 5 (mỗi lần `O(log_5 m)` ≈ 8 bước).
3. Ghép bằng bảng CRT ⇒ `C(m, i) mod 10`.
4. Cộng dồn hai tổng có trọng số, so sánh mod 10.

Tổng: `O(n log_5 n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Mô phỏng vector hóa bằng `numpy`

Vẫn `O(n²)` nhưng mỗi vòng là một phép cộng vector chạy bằng C. Nhanh hơn Python thuần nhiều lần, nhưng ở `n = 10^5` vẫn mất vài giây.

In [7]:
import numpy as np

def same_numpy(s):
    a = (np.frombuffer(s.encode(), dtype=np.uint8) - 48).astype(np.int8)
    while len(a) > 2:
        a = (a[:-1] + a[1:]) % 10
    return bool(a[0] == a[1])

for s, exp in examples:
    print(same_numpy(s), '| kỳ vọng', exp)

True | kỳ vọng True
False | kỳ vọng False


### 7.2. Cách 3 — Lucas (mod 2, mod 5) + CRT (lời giải chính)

In [8]:
SMALL5 = [[comb(a, b) % 5 for b in range(5)] for a in range(5)]

def comb_mod5(n, k):
    res = 1
    while n or k:
        a, b = n % 5, k % 5
        if b > a:
            return 0
        res = res * SMALL5[a][b] % 5
        n //= 5; k //= 5
    return res

def comb_mod10(n, k):
    return CRT10[1 if (k & n) == k else 0][comb_mod5(n, k)]

def same_lucas(s):
    d = list(map(int, s))
    m = len(d) - 2
    a = b = 0
    for i in range(m + 1):
        c = comb_mod10(m, i)
        if c:
            a += c * d[i]
            b += c * d[i + 1]
    return a % 10 == b % 10

assert all(comb_mod10(n, k) == comb(n, k) % 10 for n in range(300) for k in range(n + 1))
for s, exp in examples:
    print(same_lucas(s), '| kỳ vọng', exp)

True | kỳ vọng True
False | kỳ vọng False


### 7.3. Cách 4 — Cập nhật `C(m, i)` dần dần, tách lũy thừa 2 và 5

`C(m, i) = C(m, i−1) · (m − i + 1) / i`. Không chia được trực tiếp mod 10 vì `i` có thể chẵn hoặc chia hết cho 5. Mẹo: viết `C(m, i) = 2^{e2} · 5^{e5} · u` với `u` nguyên tố cùng nhau với 10; theo dõi `e2`, `e5` (cộng/trừ khi nhân/chia) và `u mod 10` (nhân với phần còn lại, **chia** bằng nghịch đảo mod 10 của số lẻ không chia hết cho 5: `1↔1, 3↔7, 7↔3, 9↔9`).

Giá trị mod 10: nếu `e2 > 0` và `e5 > 0` ⇒ `0`; nếu chỉ `e5 > 0` ⇒ `u · 5 ≡ 5`; nếu chỉ `e2 ≥ 0` ⇒ `u · 2^{e2} mod 10`.

In [9]:
INV10 = {1: 1, 3: 7, 7: 3, 9: 9}

def same_incremental(s):
    d = list(map(int, s))
    m = len(d) - 2
    u, e2, e5 = 1, 0, 0
    a = b = 0
    for i in range(m + 1):
        if i:
            x = m - i + 1
            while x % 2 == 0: x //= 2; e2 += 1
            while x % 5 == 0: x //= 5; e5 += 1
            u = u * (x % 10) % 10
            y = i
            while y % 2 == 0: y //= 2; e2 -= 1
            while y % 5 == 0: y //= 5; e5 -= 1
            u = u * INV10[y % 10] % 10
        if e2 and e5:
            c = 0
        elif e5:
            c = 5
        else:
            c = u * pow(2, e2, 10) % 10
        a += c * d[i]
        b += c * d[i + 1]
    return a % 10 == b % 10

for s, exp in examples:
    print(same_incremental(s), '| kỳ vọng', exp)

True | kỳ vọng True
False | kỳ vọng False


Vì sao `5^{e5} · u ≡ 5 (mod 10)` khi `e5 > 0` và `e2 = 0`: `5^{e5} ≡ 5 (mod 10)` và `u` lẻ ⇒ `5u ≡ 5`.

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 2 (`"34789"`, `m = 3`): in `C(3, i) mod 2`, `mod 5`, ghép CRT và hai tổng.

In [10]:
def trace_lucas(s):
    d = list(map(int, s))
    m = len(d) - 2
    a = b = 0
    print(f"m = {m}, chữ số = {d}")
    for i in range(m + 1):
        r2 = 1 if (i & m) == i else 0
        r5 = comb_mod5(m, i)
        c = CRT10[r2][r5]
        a += c * d[i]; b += c * d[i + 1]
        print(f"i={i}: C({m},{i})={comb(m, i)} | mod2={r2} mod5={r5} -> mod10={c} | "
              f"tổng đầu += {c}*{d[i]} -> {a}, tổng sau += {c}*{d[i + 1]} -> {b}")
    print('hai chữ số cuối:', a % 10, b % 10, '| mô phỏng:', simulate(s))

trace_lucas("34789")

m = 3, chữ số = [3, 4, 7, 8, 9]
i=0: C(3,0)=1 | mod2=1 mod5=1 -> mod10=1 | tổng đầu += 1*3 -> 3, tổng sau += 1*4 -> 4
i=1: C(3,1)=3 | mod2=1 mod5=3 -> mod10=3 | tổng đầu += 3*4 -> 15, tổng sau += 3*7 -> 25
i=2: C(3,2)=3 | mod2=1 mod5=3 -> mod10=3 | tổng đầu += 3*7 -> 36, tổng sau += 3*8 -> 49
i=3: C(3,3)=1 | mod2=1 mod5=1 -> mod10=1 | tổng đầu += 1*8 -> 44, tổng sau += 1*9 -> 58
hai chữ số cuối: 4 8 | mô phỏng: [4, 8]


## 9. Code nộp LeetCode

In [11]:
class Solution:
    def hasSameDigits(self, s: str) -> bool:
        small5 = [[1, 0, 0, 0, 0], [1, 1, 0, 0, 0], [1, 2, 1, 0, 0], [1, 3, 3, 1, 0], [1, 4, 6 % 5, 4, 1]]
        crt = [[0, 6, 2, 8, 4], [5, 1, 7, 3, 9]]          # crt[x % 2][x % 5] = x

        def c_mod5(n, k):
            r = 1
            while n or k:
                a, b = n % 5, k % 5
                if b > a:
                    return 0
                r = r * small5[a][b] % 5
                n //= 5
                k //= 5
            return r

        d = [ord(ch) - 48 for ch in s]
        m = len(d) - 2
        x = y = 0
        for i in range(m + 1):
            c = crt[1 if (i & m) == i else 0][c_mod5(m, i)]
            if c:
                x += c * d[i]
                y += c * d[i + 1]
        return x % 10 == y % 10

for s, exp in examples:
    got = Solution().hasSameDigits(s)
    print(got, 'OK' if got == exp else 'SAI')

True OK
False OK


Bảng `crt` viết tay: `crt[0] = [0, 6, 2, 8, 4]` vì `0, 6, 2, 8, 4` là các số chẵn có dư `0..4` khi chia 5; `crt[1] = [5, 1, 7, 3, 9]` cho số lẻ. Kiểm tra lại ở ô dưới.

In [12]:
assert [[0, 6, 2, 8, 4], [5, 1, 7, 3, 9]] == CRT10
print('Bảng CRT viết tay khớp bảng tính tự động.')

Bảng CRT viết tay khớp bảng tính tự động.


## 10. Vì sao thuật toán đúng

1. **Tuyến tính:** mỗi phép biến đổi là tổ hợp tuyến tính (mod 10) của các chữ số; quy nạp theo số lần biến đổi cho hệ số `C(t, j)` (hằng đẳng thức Pascal `C(t, j) = C(t−1, j−1) + C(t−1, j)`).
2. **Chỉ cần hệ số mod 10:** vì phép cộng/nhân tương thích với đồng dư.
3. **Lucas** cho `C(m, i) mod 2` và `mod 5` chính xác; **CRT** cho `mod 10` duy nhất vì `gcd(2, 5) = 1`.
4. **Cách 4:** phân tích `C(m, i) = 2^{e2} 5^{e5} u` là duy nhất; nhân/chia phần `u` là phép toán trong nhóm các số nguyên tố cùng nhau với 10 (khả nghịch mod 10), còn số mũ chỉ cộng/trừ ⇒ luôn đúng; `e2, e5 ≥ 0` vì `C(m, i)` là số nguyên.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng | `O(n²)` | `O(n)` |
| 2. `numpy` mô phỏng | `O(n²)` (vector hóa) | `O(n)` |
| 3. Lucas + CRT | `O(n log_5 n)` | `O(n)` |
| 4. Cập nhật dần, tách 2 và 5 | `O(n)` khấu hao (tổng số lần chia 2, 5 là `O(n)`) | `O(n)` |
| (tham khảo) `math.comb` số lớn | rất chậm (số hàng chục nghìn bit) | lớn |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [13]:
impls = {'numpy': same_numpy, 'lucas': same_lucas, 'incremental': same_incremental,
         'submit': lambda s: Solution().hasSameDigits(s)}
edge = [("000", True), ("123", False), ("191", True), ("55555", True), ("9" * 7, True), ("1000", False)]
for s, exp in examples + edge:
    assert same_simulate(s) == exp, (s, simulate(s))
    for name, f in impls.items():
        assert f(s) == exp, (name, s)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với mô phỏng

In [14]:
import random
rng = random.Random(3463)
cnt = 0
for _ in range(3000):
    s = ''.join(rng.choice('0123456789') for _ in range(rng.randint(3, 60)))
    exp = same_simulate(s)
    for name, f in impls.items():
        assert f(s) == exp, (name, s)
    cnt += 1
# chuỗi dài vừa, nhiều độ dài m có cấu trúc cơ số 2/5 đặc biệt
for n in [127, 128, 129, 252, 627, 628, 1002, 1250]:
    s = ''.join(rng.choice('0123456789') for _ in range(n))
    assert same_lucas(s) == same_simulate(s) == same_incremental(s)
print('Khớp mô phỏng trên', cnt, 'chuỗi ngẫu nhiên và các độ dài đặc biệt.')

Khớp mô phỏng trên 3000 chuỗi ngẫu nhiên và các độ dài đặc biệt.


### 12.3. Ca lớn `n = 10^5`

In [15]:
import time
big = {
    'ngẫu nhiên': ''.join(rng.choice('0123456789') for _ in range(10 ** 5)),
    'toàn 7': '7' * 10 ** 5,
    'n = 5^7 + 2': ''.join(rng.choice('0123456789') for _ in range(5 ** 7 + 2)),   # m = 5^7: rất nhiều C(m,i) chia hết cho 5
}
for name, s in big.items():
    t = time.time(); r1 = Solution().hasSameDigits(s); t1 = time.time() - t
    t = time.time(); r2 = same_incremental(s); t2 = time.time() - t
    assert r1 == r2
    print(f"{name:16s} n={len(s):6d}: {r1} | Lucas {t1:.2f}s | cập nhật dần {t2:.2f}s")

s = big['ngẫu nhiên'][:20000]
t = time.time(); r_np = same_numpy(s); t_np = time.time() - t
assert r_np == Solution().hasSameDigits(s)
print(f"numpy mô phỏng ở n = 20000: {t_np:.2f}s (O(n²): ở n = 10^5 cần khoảng 25 lần lâu hơn, cỡ 9–10 giây)")

ngẫu nhiên       n=100000: False | Lucas 0.05s | cập nhật dần 0.04s
toàn 7           n=100000: True | Lucas 0.05s | cập nhật dần 0.04s


n = 5^7 + 2      n= 78127: False | Lucas 0.01s | cập nhật dần 0.03s


numpy mô phỏng ở n = 20000: 0.34s (O(n²): ở n = 10^5 cần khoảng 25 lần lâu hơn, cỡ 9–10 giây)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Lấy mod 10 rồi mới chia: `C = C * (m − i) % 10 // (i + 1)`

In [16]:
def wrong_mod_then_divide(s):
    d = list(map(int, s))
    m = len(d) - 2
    c, a, b = 1, 0, 0
    for i in range(m + 1):
        a += c * d[i]; b += c * d[i + 1]
        c = c * (m - i) % 10 // (i + 1)          # SAI: phép chia không tương thích với mod
    return a % 10 == b % 10

rng = random.Random(1)
ce = next(s for s in (''.join(rng.choice('0123456789') for _ in range(rng.randint(3, 12))) for _ in range(10000))
          if wrong_mod_then_divide(s) != same_simulate(s))
print('phản ví dụ:', ce, '| sai:', wrong_mod_then_divide(ce), '| đúng:', same_simulate(ce))

phản ví dụ: 685957 | sai: False | đúng: True


**Vì sao sai:** `(x mod 10) // y` nói chung khác `(x // y) mod 10`. Phản ví dụ có độ dài 6 ⇒ hàng `m = 4`: hệ số đúng `[1, 4, 6, 4, 1]`, công thức sai cho `[1, 4, 1, 0, 0]` — ở bước tính `C(4, 2)`: `4 · 3 = 12 → 2`, rồi `2 // 2 = 1` thay vì `6`. Với vài hàng nhỏ khác công thức "tình cờ đúng" nên lỗi khó thấy nếu chỉ thử ví dụ đề.

### Lỗi 2 — Dùng nghịch đảo mod 10 cho mọi số

In [17]:
def wrong_inverse(s):
    d = list(map(int, s))
    m = len(d) - 2
    c, a, b = 1, 0, 0
    for i in range(m + 1):
        a += c * d[i]; b += c * d[i + 1]
        if i < m:
            c = c * (m - i) * pow(i + 1, -1, 10) % 10     # SAI: 2, 4, 5, 6, 8 không khả nghịch mod 10
    return a % 10 == b % 10

try:
    wrong_inverse("34789")
except ValueError as e:
    print('ValueError:', e)

ValueError: base is not invertible for the given modulus


**Vì sao sai:** chỉ các số nguyên tố cùng nhau với 10 mới có nghịch đảo mod 10. Phải tách thừa số 2 và 5 (cách 4) hoặc dùng Lucas (cách 3).

### Lỗi 3 — Dùng hàng Pascal thứ `n − 1` thay vì `n − 2`

In [18]:
def wrong_row(s):
    d = list(map(int, s))
    m = len(d) - 1                                 # SAI
    a = sum(comb_mod10(m, i) * d[i] for i in range(m)) % 10
    b = sum(comb_mod10(m, i) * d[i + 1] for i in range(m)) % 10
    return a == b

print('sai :', wrong_row("3902"), '| đúng:', Solution().hasSameDigits("3902"))

sai : False | đúng: True


**Vì sao sai:** từ `n` chữ số còn 2 chữ số cần **`n − 2`** lần biến đổi, mỗi chữ số cuối là tổ hợp của `n − 1` chữ số liên tiếp với hệ số hàng `n − 2`.

### Lỗi 4 — Ghép CRT sai: `x = (r2 + r5) mod 10` hoặc chỉ dùng `mod 5`

In [19]:
def make_wrong_crt(combine):
    def f(s):
        d = list(map(int, s))
        m = len(d) - 2
        a = b = 0
        for i in range(m + 1):
            c = combine(1 if (i & m) == i else 0, comb_mod5(m, i))
            a += c * d[i]; b += c * d[i + 1]
        return a % 10 == b % 10
    return f

for label, comb_fn in [('r2 + r5', lambda r2, r5: (r2 + r5) % 10), ('chỉ mod 5', lambda r2, r5: r5)]:
    w = make_wrong_crt(comb_fn)
    rng = random.Random(4)
    ce = next(s for s in (''.join(rng.choice('0123456789') for _ in range(rng.randint(3, 10))) for _ in range(10000))
              if w(s) != same_simulate(s))
    print(f"{label}: phản ví dụ {ce!r} | sai: {w(ce)} | đúng: {same_simulate(ce)}")

r2 + r5: phản ví dụ '388' | sai: True | đúng: False
chỉ mod 5: phản ví dụ '244519' | sai: False | đúng: True


**Vì sao sai:** CRT không phải phép cộng; phải tìm `x ∈ [0, 10)` thỏa đồng thời hai đồng dư (bảng `CRT10`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Lặp lại phép cộng kề nhau" ⇒ hệ số **nhị thức** (tam giác Pascal).
- Hệ số nhị thức **mod hợp số** ⇒ tách thành mod các lũy thừa nguyên tố (ở đây 2 và 5) + CRT; với nguyên tố dùng **Lucas**.
- `C(n, k) mod 2` ⇔ `k & n == k`.
- Khi cần chia modulo hợp số: tách các thừa số không khả nghịch ra thành số mũ riêng.

## 15. Biến thể của bài

1. **LC 3461 — bản I** (`n ≤ 100`): mô phỏng trực tiếp là đủ.
2. **LC 2221 — Find Triangular Sum of an Array:** còn **1** chữ số ⇒ hàng `n − 1`.
3. Đổi mod 10 thành mod `M` bất kỳ ⇒ phân tích `M` thành lũy thừa nguyên tố, dùng Lucas tổng quát (Granville) hoặc tách thừa số như cách 4.
4. Đếm có bao nhiêu `i` mà `C(m, i)` lẻ ⇒ `2^{popcount(m)}`.

In [20]:
# Biến thể 2 (LC 2221) bằng Lucas, và biến thể 4
def triangular_sum(nums):
    m = len(nums) - 1
    return sum(comb_mod10(m, i) * x for i, x in enumerate(nums)) % 10

def triangular_sim(nums):
    a = nums[:]
    while len(a) > 1:
        a = [(a[i] + a[i + 1]) % 10 for i in range(len(a) - 1)]
    return a[0]

rng = random.Random(15)
for _ in range(500):
    nums = [rng.randint(0, 9) for _ in range(rng.randint(1, 40))]
    assert triangular_sum(nums) == triangular_sim(nums)
assert all(sum(1 for i in range(m + 1) if (i & m) == i) == 2 ** bin(m).count('1') for m in range(500))
print('LC 2221 bằng Lucas khớp mô phỏng; số C(m,i) lẻ = 2^popcount(m). Ví dụ [1,2,3,4,5] ->', triangular_sum([1, 2, 3, 4, 5]))

LC 2221 bằng Lucas khớp mô phỏng; số C(m,i) lẻ = 2^popcount(m). Ví dụ [1,2,3,4,5] -> 8


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mô phỏng | `O(n²)` | Oracle |
| 2 | Mô phỏng `numpy` | `O(n²)` | Vài giây ở `10^5` |
| 3 | Lucas + CRT | `O(n log n)` | **Nộp bài** |
| 4 | Cập nhật dần, tách 2 và 5 | `O(n)` | Nhanh nhất, cần cẩn thận |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `s = "11111"`.

**Bài 2 (tính tay):** `C(10, 4) mod 10` bằng Lucas + CRT (so với `210`).

**Bài 3 (điền TODO):** viết `comb_mod(n, k, M)` cho `M = 6` bằng Lucas mod 2, mod 3 + CRT.

**Bài 4 (tự viết lại):** viết lại cách 4 cho modulo `M = 12 = 4 · 3` (gợi ý: tách thừa số 2 và 3).

**Bài 5 (mini challenge):** với `n = 10^5`, đếm có bao nhiêu `i` mà `C(n − 2, i) ≡ 0 (mod 10)`.

In [21]:
def comb_mod6_todo(n, k):
    # TODO
    raise NotImplementedError

def same_mod12_todo(s):
    # TODO
    raise NotImplementedError

def count_zero_coeffs_todo(n):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Mọi chữ số bằng nhau. (2) Sau mỗi bước vẫn bằng nhau. (3) Kết quả `True`.

**Bài 2.** (1) `10 = (1010)_2`, `4 = (0100)_2` ⇒ mod 2. (2) `10 = (20)_5`, `4 = (04)_5` ⇒ mod 5. (3) Tra bảng CRT.

**Bài 3.** (1) `comb_mod_p(n, k, 2)` và `comb_mod_p(n, k, 3)`. (2) Bảng CRT 6 ô. (3) Kiểm tra với `math.comb`.

**Bài 4.** (1) `C = 2^{e2} 3^{e3} u` với `gcd(u, 6) = 1`. (2) Nghịch đảo mod 12 của `1, 5, 7, 11` là chính nó. (3) Giá trị: `u · 2^{e2} · 3^{e3} mod 12`.

**Bài 5.** (1) `C ≡ 0 (mod 10)` ⇔ chẵn **và** chia hết cho 5. (2) Chẵn ⇔ `i & m != i`. (3) Chia hết cho 5 ⇔ có chữ số cơ số 5 của `i` lớn hơn của `m`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2
assert Solution().hasSameDigits("11111") is True
r2 = 1 if (4 & 10) == 4 else 0
r5 = comb_mod5(10, 4)
assert CRT10[r2][r5] == comb(10, 4) % 10 == 0
print('Bài 1: True | Bài 2: mod2 =', r2, 'mod5 =', r5, '-> mod10 =', CRT10[r2][r5], '(210)')

# Bài 3
CRT6 = [[None] * 3 for _ in range(2)]
for x in range(6):
    CRT6[x % 2][x % 3] = x

def comb_mod6(n, k):
    return CRT6[comb_mod_p(n, k, 2)][comb_mod_p(n, k, 3)]

assert all(comb_mod6(n, k) == comb(n, k) % 6 for n in range(200) for k in range(n + 1))
print('Bài 3: comb_mod6 khớp math.comb với n < 200.')

Bài 1: True | Bài 2: mod2 = 0 mod5 = 0 -> mod10 = 0 (210)
Bài 3: comb_mod6 khớp math.comb với n < 200.


In [23]:
# Bài 4 — mod 12, và Bài 5
def same_mod12(s):
    d = list(map(int, s))
    m = len(d) - 2
    u, e2, e3 = 1, 0, 0
    a = b = 0
    for i in range(m + 1):
        if i:
            x = m - i + 1
            while x % 2 == 0: x //= 2; e2 += 1
            while x % 3 == 0: x //= 3; e3 += 1
            u = u * x % 12
            y = i
            while y % 2 == 0: y //= 2; e2 -= 1
            while y % 3 == 0: y //= 3; e3 -= 1
            u = u * pow(y, -1, 12) % 12
        c = u * pow(2, e2, 12) * pow(3, e3, 12) % 12
        a += c * d[i]; b += c * d[i + 1]
    return a % 12 == b % 12

def simulate_mod(s, M):
    a = list(map(int, s))
    while len(a) > 2:
        a = [(a[i] + a[i + 1]) % M for i in range(len(a) - 1)]
    return a[0] == a[1]

rng = random.Random(19)
for _ in range(1000):
    s = ''.join(rng.choice('0123456789') for _ in range(rng.randint(3, 40)))
    assert same_mod12(s) == simulate_mod(s, 12)
print('Bài 4 khớp mô phỏng mod 12.')

def count_zero_coeffs(n):
    m = n - 2
    return sum(1 for i in range(m + 1) if (i & m) != i and comb_mod5(m, i) == 0)

assert count_zero_coeffs(40) == sum(1 for i in range(39) if comb(38, i) % 10 == 0)
print('Bài 5: n = 10^5 có', count_zero_coeffs(10 ** 5), 'hệ số chia hết cho 10 trên', 10 ** 5 - 1)

Bài 4 khớp mô phỏng mod 12.
Bài 5: n = 10^5 có 79691 hệ số chia hết cho 10 trên 99999


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
m = n - 2
crt = [[0, 6, 2, 8, 4], [5, 1, 7, 3, 9]]          # crt[x%2][x%5] = x
c(i) = crt[(i & m) == i][lucas_mod5(m, i)]
a = Σ c(i) * d[i];  b = Σ c(i) * d[i+1]
return a % 10 == b % 10
```

**Checklist:**

- [ ] Hàng Pascal `m = n − 2`?
- [ ] Không chia sau khi đã lấy mod 10?
- [ ] Lucas mod 5: chữ số `k` lớn hơn chữ số `n` ⇒ 0?
- [ ] CRT bằng bảng, không cộng `r2 + r5`?
- [ ] Không dùng `math.comb` với `m ~ 10^5` (quá chậm)?